<div class="alert alert-info">
    <strong>导航</strong>
    <ul>
        <li>上一节： <a href="9_23_observing_design_and_archive_reanalysis.ipynb">9.23 观测设计与归档数据再分析</a></li>
        <li>下一节： <a href="9_x_further_reading_and_workflow.ipynb">9.x 延伸阅读与后续实践方向</a></li>
    </ul>
</div>


## 9.24 软件生态与可复现实践：从工具链到 Provenance

现代射电干涉数据处理很少由单一软件完成。一个项目可能用 CASA 导入和校准，用 WSClean 或 DDFacet 成像，用 CARTA 检查 cube 和残差，用 PyBDSF 生成连续谱源表，用 SoFiA 搜索谱线源，用 Astropy、radio-beam、spectral-cube 和 regions 做科学测量，最后用脚本、notebook 和参数文件组织结果。若把这些工具看成互相独立的命令集合，处理流程很快会失去可追溯性；若把它们看成测量链中的不同层级，就能清楚判断每个工具改变了数据、模型、权重、坐标、单位还是解释。

本节的目标不是写软件手册，而是建立软件生态的教材视角：工具解决哪一类问题，输出产品携带哪些假设，哪些信息必须作为 provenance 保存，哪些结果需要独立验证。射电数据处理的可复现性不是“把 notebook 发出去”这么简单，而是让输入数据、软件版本、参数、日志、中间产品、人工判断和最终科学量能够被重新连接起来。


### 9.24.1 工具链分层：软件名不是工作流结构

射电数据处理可以按层级理解。第一层是数据容器与元数据，例如 Measurement Set、ASDM、FITS-IDI、FITS image 和 catalogue table。第二层是 flagging、averaging、calibration 和 imaging，它们改变可见度、权重、校准表、模型图和残差图。第三层是可视化和诊断，用来决定数据是否可信、mask 是否合理、残差是否有系统结构。第四层是源搜索、谱线检测、区域测量和科学量转换。第五层是 provenance 和 validation，贯穿所有步骤。

这种分层比软件名本身更重要。CASA 可以做导入、flag、校准、成像和测量；WSClean 主要面向高性能成像；DP3 常用于 LOFAR/低频数据的预处理和校准链；DDFacet/killMS 面向方向相关成像和校准；CARTA 是交互式可视化和检查工具；PyBDSF 是连续谱图像到源表的工具；SoFiA 是谱线 cube 中三维源搜索和可靠性评估工具；Astropy 生态承担坐标、单位、FITS、区域、beam 和可复现测量。一个软件可以跨多个层级，但一个科学结果不能只靠软件名说明其可靠性。


![toolchain layers](figures/practical_software_ecosystem_toolchain_layers.png)

图中把实践工作流分成数据、flag/calibration、imaging、visual inspection、catalogue/source finding、science measurement、provenance 和 validation 等层。PyBDSF 被放在连续谱图像到源表的层级，SoFiA 被放在谱线 cube 三维 source finding 层级，两者解决的问题不同，不应混用。CARTA 不替代测量脚本，但它非常适合发现 mask、残差、速度结构和主波束边缘问题。


### 9.24.2 数据格式转换会改变可追溯对象

从原始望远镜数据到 Measurement Set，再到校准表、图像、cube、源表和论文图，每一次格式转换都会改变需要记录的信息。Measurement Set 不只是一个 visibility 数组，还包含 FIELD、SPECTRAL_WINDOW、ANTENNA、DATA_DESCRIPTION、FLAG、WEIGHT 等表。FITS 图像不只是像素矩阵，还包含 WCS、beam、单位、频率轴、Stokes 轴和历史记录。源表不只是坐标列表，还包含阈值、局部 RMS、island/component/source 关系、拟合模型和质量标记。

格式转换的风险来自隐式假设。单位可能从 Jy/beam 变成 Jy/pixel 或 K；速度轴可能从 frequency 解释成 radio velocity；flag 和 weight 可能在导出时丢失；CASA image 转 FITS 后可能只保留部分历史；region 文件可能依赖某个坐标系；notebook 中的本机绝对路径会使复现失败。因此，转换本身也应被当作处理步骤记录，而不是无害的文件整理。


![format conversion](figures/practical_software_data_format_conversion.png)

图中展示从原始数据、Measurement Set、校准表、图像产品、分析表到 provenance record 的关系。每一个箭头都可能改变数据含义。可靠工作流会记录输入文件身份、转换命令、输出单位、坐标、beam、权重和版本，而不是只保存最后一张 FITS 图。


### 9.24.3 Provenance：结果必须带着来源移动

Provenance 指结果从哪些输入、经过哪些软件、哪些参数和哪些人工判断得到。对射电干涉而言，最低限度的 provenance 应包括：项目编号或数据集标识；原始数据版本；flag 版本；校准表和 apply 顺序；成像参数；mask 和 region 文件；软件版本；Python 环境；关键日志；最终图像、模型、残差和源表；已知失败模式和人工判断。若结果来自 pipeline，还应记录 pipeline 版本、weblog、产品类型和是否做过后处理。

参数文件比散落在 notebook 文本中的参数更可靠。YAML、JSON、parset、CASA script、WSClean command file 或 Snakemake/Makefile 都可以承担这个角色。重要的是，参数和结果之间有稳定连接。一个结果若只能通过“当时在交互界面里点了几下”解释，就很难复现；一个结果若能用固定输入、固定参数和固定软件环境重新生成，即使软件内部算法复杂，也更容易被审查和教学。


![provenance bundle](figures/practical_software_provenance_bundle.png)

图中把可复现结果看成一个 bundle：数据身份、软件版本、参数、日志、中间产品和人工判断共同指向最终图像、表格或科学量。任何一个部分缺失，结果都可能仍然“看起来正确”，但难以解释为什么与另一次处理不同。


### 9.24.4 Notebook 的角色：解释和测量，而不是隐藏状态

Notebook 很适合教材、探索和小型可复现测量，因为文字、公式、图和代码可以放在一起。但 notebook 也容易隐藏状态：单元执行顺序不明、变量来自旧运行、绝对路径写死、交互式修改没有记录、输出文件被覆盖、环境依赖没有保存。对于大型数据处理，notebook 更适合承担解释、诊断和最终测量角色；长时间运行的校准和成像步骤更适合放在脚本、参数文件或 workflow engine 中。

可移植 notebook 应使用相对路径或项目根目录变量，参数集中在配置文件中，输入输出文件名显式可见，随机种子和软件版本被记录，交互式 region 或 mask 保存为文件，最终单元包含基本验证，例如输入是否存在、beam 是否匹配、单位是否正确、源表行数是否在预期范围内。这样 notebook 才能成为教材和研究之间的桥，而不是个人机器状态的截图。


![notebook patterns](figures/practical_software_reproducible_notebook_patterns.png)

图中对比了脆弱 notebook 和可移植 workflow。问题通常不在 notebook 形式本身，而在是否把数据路径、参数、环境、人工选择和输出验证显式化。对于教材而言，notebook 应尽量让每一步的物理意义和统计意义可见，而不是把复杂处理藏在不可追踪的交互操作中。


### 9.24.5 Validation：软件输出不是科学结论

软件可以产生图像、cube、源表、RM 图或 moment 图，但这些产品还需要 validation 才能变成科学结果。连续谱源表需要检查局部 RMS、残差、亮源旁瓣、注入源恢复率、虚假检测率和多分量合并。谱线 source finding 需要检查三维 mask、可靠性、噪声相关、通道边缘和全局谱线。图像测量需要检查 beam、单位、背景、相关噪声和主波束。方向相关成像需要检查残差是否按方向、频率和时间系统变化。

Validation 最好形成闭环。处理输出先产生诊断图和质量指标；若诊断失败，回到 flag、校准、权重、mask 或成像参数；若诊断通过，再进入科学测量；测量结果还需要独立复现或至少在新环境中重跑关键脚本。对教学而言，失败案例和诊断图比成功命令更有价值，因为真实数据能力往往来自识别结果何时不可信。


![validation loop](figures/practical_software_validation_loop.png)

图中把软件输出、诊断图、定量检查、科学测量、独立复现和发布包连成闭环。这个闭环可以很轻量，也可以由完整 workflow engine 管理。关键是软件产品不能直接等同于科学结论，中间必须有验证和误差预算。


### 9.24.6 与真实软件工作流的对应

一个实际项目可以采用轻量但规范的目录结构：`data/` 保存输入或软链接，`configs/` 保存参数文件，`scripts/` 保存可重复运行的步骤，`logs/` 保存命令输出，`products/` 保存图像和源表，`notebooks/` 保存解释和测量，`figures/` 保存论文或教材图。版本控制适合保存脚本、配置、文本日志和小型图表；大型 Measurement Set、image cube 和中间产品通常需要用数据仓库、对象存储、校验和或明确下载说明管理，而不是直接塞进 Git。

在 CASA 工作流中，`casalog`、task 参数、calibration tables、flag versions 和 image history 都应被保存。WSClean 工作流应保存完整命令、权重、multiscale/multifrequency 设置和输出模型/残差。DDFacet/killMS 或 DP3 工作流应保存 parset、solution tables、方向划分和 beam 模型。PyBDSF 应保存 parameter file、catalogue、island/component/source 层级产品和 residual/model image。SoFiA 应保存 mask、reliability table、moment products 和 source catalogue。Python 测量应保存 environment、region、unit conversion 和表格 schema。


### 9.24.7 本节结论

软件生态的核心不是掌握更多命令，而是知道每个工具处在测量链的哪一层，以及它改变了哪些对象。CASA、WSClean、DP3、DDFacet、CARTA、PyBDSF、SoFiA 和 Astropy 生态各有位置；把它们串起来的是数据模型、参数、日志、版本、验证和 provenance。

可复现实践并不要求所有项目都变成大型自动化 pipeline，但要求最终结果能够回答三个问题：输入是什么，处理如何进行，为什么认为输出可信。只要这三个问题有清晰记录，教材案例就能从“照着命令跑一次”转化为真正可迁移的科研训练。
